# 07. Out-of-Sample Test

Este notebook evalúa el rendimiento fuera de muestra (Out-of-Sample / OOS) de los modelos congelados (frozen models) entrenados previamente, garantizando la ausencia de sesgos de anticipación (look-ahead bias) o filtración de datos (data leakage).

El objetivo fundamental es auditar la capacidad predictiva y la consistencia estadística de las señales generadas en un entorno no visto. A través del análisis del Rank Information Coefficient (Rank IC), la estabilidad temporal de la señal, la precisión direccional (Hit Rate) y la monotonía por deciles, se dictaminará si el modelo conserva su robustez cuantitativa antes de proceder a la fase de simulación de carteras y backtesting.


## 1. Imports & Configuration

### 1.1 Librerias

In [1]:
import sys
import pandas as pd 
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import random
import warnings
import joblib

from pathlib import Path

# adds the root carpet of the project to the sys.path to allow importing modules from src
sys.path.append(str(Path.cwd().parent))

# autoreload configuration to automatically reload modules when they are modified
%load_ext autoreload
%autoreload 2

In [2]:
# =============================================================================
# Reproducibility
# =============================================================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)


# =============================================================================
# Warnings
# =============================================================================

warnings.filterwarnings(
    "ignore",
    category=FutureWarning,
)

warnings.filterwarnings(
    "ignore",
    category=UserWarning,
)

## 2. Load Frozen Model & Configuration


### 2.1 Load final model

In [3]:
final_ridge_rank = joblib.load(
    "../data/model_results/final_model/final_ridge_rank.joblib"
)

final_xgb_rank = joblib.load(
    "../data/model_results/final_model/final_xgb_rank.joblib"
)

final_rf_rank = joblib.load(
    "../data/model_results/final_model/final_rf_rank.joblib"
)

print("Final models loaded successfully.")

print(f"Ridge:        {type(final_ridge_rank).__name__}")
print(f"XGBoost:      {type(final_xgb_rank).__name__}")
print(f"Random Forest:{type(final_rf_rank).__name__}")

Final models loaded successfully.
Ridge:        Ridge
XGBoost:      XGBRegressor
Random Forest:RandomForestRegressor


### 2.2 Load model metadata

In [4]:
import json

with open(
    "../data/model_results/final_model/final_model_configurations.json",
    "r",
    encoding="utf-8",
) as f:

    final_model_configurations = json.load(f)

print("Final model configurations loaded successfully.")

Final model configurations loaded successfully.


### 2.3 Verify model configuration

In [5]:
with open(
    "../data/model_results/final_model/training_metadata.json",
    "r",
    encoding="utf-8",
) as f:

    training_metadata = json.load(f)

print("Training metadata loaded successfully.")

Training metadata loaded successfully.


### 2.4 Verify frozen configuration

In [6]:
# -----------------------------------------------------------------------------
# Verify loaded models
# -----------------------------------------------------------------------------

print("\nModels loaded:")

print(
    f"  Ridge:         {type(final_ridge_rank).__name__}"
)

print(
    f"  XGBoost:       {type(final_xgb_rank).__name__}"
)

print(
    f"  Random Forest: {type(final_rf_rank).__name__}"
)


# -----------------------------------------------------------------------------
# Verify model configurations
# -----------------------------------------------------------------------------

print("\nFrozen model configurations:")

for model_name, configuration in final_model_configurations.items():

    print(f"\n{model_name}:")
    
    if isinstance(configuration, dict):

        for parameter, value in configuration.items():

            print(
                f"  {parameter:<25}: {value}"
            )

    else:

        print(
            f"  {configuration}"
        )


# -----------------------------------------------------------------------------
# Verify training metadata
# -----------------------------------------------------------------------------

print("\nTraining metadata:")

if isinstance(training_metadata, dict):

    for section, values in training_metadata.items():

        print(f"\n[{section}]")

        if isinstance(values, dict):

            for key, value in values.items():

                print(
                    f"  {key:<25}: {value}"
                )

        else:

            print(
                f"  {values}"
            )


print("\n" + "=" * 80)
print("FROZEN CONFIGURATION VERIFICATION COMPLETED")
print("=" * 80)


Models loaded:
  Ridge:         Ridge
  XGBoost:       XGBRegressor
  Random Forest: RandomForestRegressor

Frozen model configurations:

XGBoost:
  representation           : Percentile Rank
  features                 : ['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']
  target                   : forward_return_21d
  hyperparameters          : {'learning_rate': 0.05143828405076928, 'max_depth': 4, 'min_child_weight': 9.726261649881026, 'subsample': 0.9100531293444458, 'colsample_bytree': 0.9757995766256756, 'gamma': 4.474136752138244, 'reg_alpha': 0.09761125443110447, 'reg_lambda': 4.869640941520899}
  random_state             : 42

Ridge:
  representation           : Percentile Rank
  features                 : ['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']
  target                   : forward_return_21d
  hyperparameters          : {'alpha': 506.1576888752306}
  random_state             : 42

Random Forest:
  repre

In [7]:
# =============================================================================
# Frozen Experiment Definition
# =============================================================================

EXPECTED_FEATURES = [
    "momentum_12_1_win_rank",
    "upside_volatility_win_rank",
    "log10_amihud_win_rank",
]

EXPECTED_REPRESENTATION = "Percentile Rank"
EXPECTED_TARGET = "forward_return_21d"
EXPECTED_HORIZON = 21
EXPECTED_SEED = 42


print("\n" + "=" * 80)
print("FROZEN EXPERIMENT DEFINITION")
print("=" * 80)

print("\nFeatures:")
for feature in EXPECTED_FEATURES:
    print(f"  - {feature}")

print(f"\nRepresentation: {EXPECTED_REPRESENTATION}")
print(f"Target:         {EXPECTED_TARGET}")
print(f"Horizon:        {EXPECTED_HORIZON} trading days")
print(f"Seed:            {EXPECTED_SEED}")

print("\n" + "=" * 80)


FROZEN EXPERIMENT DEFINITION

Features:
  - momentum_12_1_win_rank
  - upside_volatility_win_rank
  - log10_amihud_win_rank

Representation: Percentile Rank
Target:         forward_return_21d
Horizon:        21 trading days
Seed:            42



## 3. Load OOS Dataset

El período Out-of-Sample se define de forma estrictamente posterior al conjunto utilizado durante el desarrollo y la selección de los modelos. Dado que la variable objetivo corresponde al retorno acumulado a 21 sesiones (forward_return_21d), las últimas 21 sesiones del período de precios disponible no pueden utilizarse como observaciones de desarrollo, ya que no existe información suficiente para calcular su retorno futuro completo. Por este motivo, aunque los precios disponibles alcanzan el 30 de diciembre de 2024, el último target válido del período de desarrollo se sitúa aproximadamente a principios de diciembre de 2024.

A partir de esta fecha se mantiene un período adicional de separación temporal, siguiendo el principio de las ventanas de purging y embargo empleado anteriormente en el esquema CPCV. Esta separación evita que exista solapamiento entre los horizontes de los retornos utilizados durante el desarrollo y las observaciones destinadas al test final. De este modo, el período Out-of-Sample comienza aproximadamente a mediados de enero de 2025.

### 3.1 Load OOS features

En esta sección se construye el conjunto de variables explicativas que será utilizado posteriormente para evaluar el modelo sobre datos Out-of-Sample (OOS). El objetivo es reproducir exactamente el mismo proceso de construcción y transformación utilizado durante la fase de desarrollo del modelo, pero aplicándolo exclusivamente a información posterior al período de Train/Validation.

In [ ]:
# =============================================================================
# Extended S&P 500 Price Dataset for OOS Testing
# =============================================================================

import pandas as pd

from src.data.download_data import (
    download_prices,
    remove_empty_tickers,
    validate_download,
)

# =============================================================================
# Configuration
# =============================================================================

START_DATE = "2010-01-01"

# OOS evaluation period ends on 15/07/2026.
# Additional data is required afterwards to calculate the 21-day forward return
# for the final OOS observations.
END_DATE = "2026-09-01"

INTERVAL = "1d"
AUTO_ADJUST = False

# =============================================================================
# S&P 500 Constituents
# =============================================================================

url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

sp500 = pd.read_html(
    url,
    storage_options={"User-Agent": "Mozilla/5.0"},
)[0]

# Yahoo Finance uses "-" instead of "." in ticker symbols

tickers = (
    sp500["Symbol"]
    .str.replace(".", "-", regex=False)
    .tolist()
)

print(f"Requested tickers: {len(tickers)}")

# =============================================================================
# Download prices
# =============================================================================

prices = download_prices(
    tickers=tickers,
    START_DATE=START_DATE,
    END_DATE=END_DATE,
    INTERVAL=INTERVAL,
    AUTO_ADJUST=AUTO_ADJUST,
)

# =============================================================================
# Remove completely empty tickers
# =============================================================================

prices = remove_empty_tickers(
    prices
)

# =============================================================================
# Datetime and MultiIndex standardization
# =============================================================================

prices.index = pd.to_datetime(
    prices.index
)

prices.columns = pd.MultiIndex.from_tuples(
    [
        (str(c[0]), str(c[1]))
        for c in prices.columns
    ],
    names=["Price", "Ticker"],
)

# =============================================================================
# Validate downloaded data
# =============================================================================

validate_download(
    prices,
    tickers,
)

# =============================================================================
# Final dataset checks
# =============================================================================

print("\n" + "=" * 50)
print("EXTENDED PRICE DATASET")
print("=" * 50)

print(
    f"Start date : {prices.index.min().date()}"
)

print(
    f"End date   : {prices.index.max().date()}"
)

print(
    f"Rows       : {len(prices):,}"
)

print(
    f"Tickers    : "
    f"{prices.columns.get_level_values('Ticker').nunique():,}"
)

# =============================================================================
# Save extended dataset
# =============================================================================

prices.to_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)


print(
    "\nSaved to:"
    "\n../data/raw/sp500_prices_extended.parquet"
)

Requested tickers: 503


[*********************100%***********************]  503 of 503 completed


DOWNLOAD SUMMARY
Requested tickers : 503
Valid downloads   : 503
Missing tickers   : 0
Empty tickers (NaN): 0

All tickers downloaded and validated successfully.

EXTENDED PRICE DATASET
Start date : 2010-01-04
End date   : 2026-08-10
Rows       : 4,175
Tickers    : 503

Saved to:
../data/raw/sp500_prices_extended.parquet


Las variables utilizadas son los tres factores seleccionados durante la fase de modelización: 12-1 Momentum, Upside Volatility y log10 Amihud Illiquidity. Estos factores se calculan directamente a partir de los precios y volúmenes contenidos en el dataset extendido de mercado. Posteriormente, se aplica el mismo procedimiento de winsorización cross-sectional al 1%-99% y de percentile ranking cross-sectional, centrado alrededor de cero, empleado durante el desarrollo del modelo.

In [13]:
from src.preprocessing.factor_preprocessing import (
    winsorize_cross_sectional,
    rank_cross_sectional,
)

from src.features.liquidity import (
    compute_amihud_illiquidity,
)

from src.features.momentum import (
    compute_12_1_momentum,
)

from src.features.volatility import (
    compute_upside_volatility,
)


# =============================================================================
# OOS Evaluation Period
# =============================================================================

OOS_START = pd.Timestamp("2025-01-15")
OOS_END = pd.Timestamp("2026-07-15")


# =============================================================================
# Load extended price data
# =============================================================================

prices = pd.read_parquet(
    "../data/raw/sp500_prices_extended.parquet"
)


# =============================================================================
# Remove securities excluded during development
# =============================================================================

tickers_to_remove = [
    "SW",
    "AMCR",
]

prices = prices.drop(
    columns=tickers_to_remove,
    level=1,
    errors="ignore",
)


# =============================================================================
# Verify price data coverage
# =============================================================================

print("=" * 80)
print("PRICE DATA COVERAGE")
print("=" * 80)

print(
    f"\nAvailable price period: "
    f"{prices.index.min().date()} → {prices.index.max().date()}"
)

print(
    f"OOS evaluation period: "
    f"{OOS_START.date()} → {OOS_END.date()}"
)

assert prices.index.max() > OOS_END, (
    "Price data does not extend beyond OOS_END. "
    "Additional observations are required to compute the "
    "21-day forward return."
)


PRICE DATA COVERAGE

Available price period: 2010-01-04 → 2026-08-10
OOS evaluation period: 2025-01-15 → 2026-07-15


In [15]:
# =============================================================================
# Compute returns
# =============================================================================

adj_close = prices["Adj Close"]

simple_returns = adj_close.pct_change()

log_returns = np.log(
    adj_close / adj_close.shift(1)
)

In [16]:
# =============================================================================
# Compute selected factors
# =============================================================================

# -----------------------------------------------------------------------------
# 12-1 Momentum
# -----------------------------------------------------------------------------

momentum_12_1 = compute_12_1_momentum(
    prices
)


# -----------------------------------------------------------------------------
# Upside Volatility
# -----------------------------------------------------------------------------

upside_volatility = compute_upside_volatility(
    log_returns
)


# -----------------------------------------------------------------------------
# Amihud Illiquidity
# -----------------------------------------------------------------------------

amihud_illiquidity = compute_amihud_illiquidity(
    prices=prices,
    simple_returns=simple_returns,
)

log10_amihud = np.log10(
    amihud_illiquidity
)


# =============================================================================
# Combine factor series into a single panel DataFrame
# =============================================================================

factor_matrix = (
    pd.concat(
        {
            "momentum_12_1": momentum_12_1.stack(),
            "upside_volatility": upside_volatility.stack(),
            "log10_amihud": log10_amihud.stack(),
        },
        axis=1,
    )
    .rename_axis(["date", "ticker"])
    .reset_index()
)


# =============================================================================
# Cross-sectional winsorization
# =============================================================================

factor_cols = [
    "momentum_12_1",
    "upside_volatility",
    "log10_amihud",
]

df_win = winsorize_cross_sectional(
    factor_matrix,
    cols=factor_cols,
    p_low=0.01,
    p_high=0.99,
)


# =============================================================================
# Cross-sectional percentile rank
# =============================================================================

win_cols = [
    f"{col}_win"
    for col in factor_cols
]

df_rank = rank_cross_sectional(
    df_win,
    cols=win_cols,
)


# =============================================================================
# Build OOS feature matrix
# =============================================================================

X_oos = (
    df_rank[
        [
            "date",
            "ticker",
            "momentum_12_1_win_rank",
            "upside_volatility_win_rank",
            "log10_amihud_win_rank",
        ]
    ]
    .set_index(["date", "ticker"])
)


print("\nOOS features constructed successfully.")


OOS features constructed successfully.


Una consideración importante es que el universo de securities utilizado durante el OOS debe ser consistente con el universo empleado durante la fase de desarrollo. Por este motivo, no se utiliza directamente el universo actual del S&P 500 ni se introducen nuevos componentes que no estuvieran presentes en Train/Validation. El universo de referencia se recupera del dataset persistente df_final_rank.parquet, utilizado durante el desarrollo, y las observaciones OOS se restringen a dicho conjunto de securities. De esta forma, cualquier diferencia observada posteriormente puede atribuirse al comportamiento del modelo sobre un período temporal nuevo y no a un cambio en el universo de activos evaluado.

El período OOS se ha fijado explícitamente entre el 15 de enero de 2025 y el 15 de julio de 2026. Esta delimitación se mantiene fija para garantizar la reproducibilidad del experimento y evitar que el período de evaluación se amplíe automáticamente cada vez que se ejecute el notebook con datos de mercado más recientes.

In [18]:
# =============================================================================
# Load development dataset
# =============================================================================

df_rank_dev = pd.read_parquet(
    "../data/preprocessed/df_final_rank.parquet"
)

# =============================================================================
# Recover development universe
# =============================================================================

development_tickers = (
    df_rank_dev.index
    .get_level_values("ticker")
    .unique()
)


print("=" * 80)
print("UNIVERSE CONSISTENCY CHECK")
print("=" * 80)

print(
    f"\nDevelopment universe: "
    f"{len(development_tickers):,} tickers"
)



UNIVERSE CONSISTENCY CHECK

Development universe: 497 tickers


In [19]:
# =============================================================================
# Compare OOS and development universes
# =============================================================================

oos_tickers_before = set(
    X_oos.index
    .get_level_values("ticker")
    .unique()
)

development_tickers_set = set(
    development_tickers
)


new_oos_tickers = (
    oos_tickers_before
    - development_tickers_set
)

common_tickers = (
    oos_tickers_before
    & development_tickers_set
)


print(
    f"OOS tickers before filtering: "
    f"{len(oos_tickers_before):,}"
)

print(
    f"Common tickers: "
    f"{len(common_tickers):,}"
)

print(
    f"New OOS-only tickers removed: "
    f"{len(new_oos_tickers):,}"
)

if new_oos_tickers:
    print("\nOOS-only tickers:")
    print(sorted(new_oos_tickers))

OOS tickers before filtering: 501
Common tickers: 496
New OOS-only tickers removed: 5

OOS-only tickers:
['FDXF', 'FERG', 'HONA', 'Q', 'SNDK']


In [20]:
# =============================================================================
# Restrict OOS to development universe
# =============================================================================

oos_ticker_mask = (
    X_oos.index
    .get_level_values("ticker")
    .isin(development_tickers)
)

X_oos = X_oos.loc[oos_ticker_mask]


# =============================================================================
# Restrict to OOS evaluation period
# =============================================================================

oos_dates = (
    X_oos.index
    .get_level_values("date")
)

date_mask = (
    (oos_dates >= OOS_START)
    & (oos_dates <= OOS_END)
)

X_oos = X_oos.loc[date_mask]

Finalmente, se realizan comprobaciones de integridad sobre el período temporal, el número de securities y observaciones, la ausencia de valores faltantes y el rango de las variables transformadas. Las 102 observaciones que presentaban información incompleta en alguna de las tres variables fueron excluidas, quedando un dataset final de 185.898 observaciones correspondientes a 496 securities, sin valores faltantes.

El resultado de este proceso es X_oos, una matriz de características completamente independiente de las predicciones del modelo y preparada para ser utilizada como entrada del modelo final congelado en la siguiente etapa del análisis OOS.

In [21]:
# =============================================================================
# Missing-value diagnostics
# =============================================================================

print("\n" + "=" * 80)
print("OOS FEATURE DATASET — BEFORE CLEANING")
print("=" * 80)

oos_dates = (
    X_oos.index
    .get_level_values("date")
)

print(
    f"\nOOS period: "
    f"{oos_dates.min()} → {oos_dates.max()}"
)

print(
    f"Observations: "
    f"{len(X_oos):,}"
)

print(
    f"Tickers: "
    f"{X_oos.index.get_level_values('ticker').nunique():,}"
)

print("\nFeatures:")
print(X_oos.columns.tolist())

print("\nMissing values:")
print(X_oos.isna().sum())

print("\nMissing percentage:")
print(
    (X_oos.isna().mean() * 100).round(3)
)


OOS FEATURE DATASET — BEFORE CLEANING

OOS period: 2025-01-15 00:00:00 → 2026-07-15 00:00:00
Observations: 186,000
Tickers: 496

Features:
['momentum_12_1_win_rank', 'upside_volatility_win_rank', 'log10_amihud_win_rank']

Missing values:
momentum_12_1_win_rank        102
upside_volatility_win_rank      0
log10_amihud_win_rank           0
dtype: int64

Missing percentage:
momentum_12_1_win_rank        0.055
upside_volatility_win_rank    0.000
log10_amihud_win_rank         0.000
dtype: float64


In [ ]:
# =============================================================================
# Identify observations with incomplete features
# =============================================================================

incomplete_mask = (
    X_oos.isna()
    .any(axis=1)
)

print(
    f"\nObservations with incomplete features: "
    f"{incomplete_mask.sum():,}"
)

print(
    f"Observations with complete features: "
    f"{(~incomplete_mask).sum():,}"
)

# =============================================================================
# Remove incomplete observations
# =============================================================================

X_oos = X_oos.loc[
    ~incomplete_mask
]


# =============================================================================
# Final dataset diagnostics
# =============================================================================

print("\n" + "=" * 80)
print("OOS FEATURE DATASET — FINAL")
print("=" * 80)

oos_dates = (
    X_oos.index
    .get_level_values("date")
)

oos_tickers = (
    X_oos.index
    .get_level_values("ticker")
)


print(
    f"\nOOS period: "
    f"{oos_dates.min()} → {oos_dates.max()}"
)

print(
    f"Observations: "
    f"{len(X_oos):,}"
)

print(
    f"Tickers: "
    f"{oos_tickers.nunique():,}"
)

print("\nMissing values:")
print(X_oos.isna().sum())

print("\nFeature ranges:")
print(
    X_oos.describe().loc[
        ["min", "max"]
    ]
)


# =============================================================================
# Final assertions
# =============================================================================

assert len(X_oos) > 0, (
    "OOS feature dataset is empty. "
    "Check OOS_START, OOS_END, universe and price data coverage."
)

assert oos_dates.min() >= OOS_START

assert oos_dates.max() <= OOS_END

assert (
    set(oos_tickers)
    .issubset(development_tickers_set)
), (
    "OOS dataset contains tickers that were not present "
    "in the development universe."
)

assert not X_oos.isna().any().any(), (
    "Missing values remain in the final OOS feature dataset."
)


print("\n" + "=" * 80)
print("OOS FEATURE DATASET READY")
print("=" * 80)


Observations with incomplete features: 102
Observations with complete features: 185,898

OOS FEATURE DATASET — FINAL

OOS period: 2025-01-15 00:00:00 → 2026-07-15 00:00:00
Observations: 185,898
Tickers: 496

Missing values:
momentum_12_1_win_rank        0
upside_volatility_win_rank    0
log10_amihud_win_rank         0
dtype: int64

Feature ranges:
     momentum_12_1_win_rank  upside_volatility_win_rank  log10_amihud_win_rank
min               -0.493976                   -0.493988                 -0.494
max                0.495984                    0.495992                  0.496

OOS FEATURE DATASET READY


### 3.2 Load OOS forward returns

Una vez construidas y validadas las variables explicativas OOS, se construye el target que permitirá evaluar las predicciones del modelo sobre datos no utilizados durante su desarrollo. Se utiliza la misma definición de `forward_return_21d` empleada durante la fase de Train/Validation, calculando para cada observación el retorno acumulado de los **21 días de negociación posteriores** a la fecha de referencia.

El cálculo se realiza directamente sobre los precios ajustados del dataset extendido, que contiene información posterior al período de evaluación. Esto permite disponer de los retornos futuros necesarios para evaluar incluso las últimas observaciones del período OOS.

Aunque el período de evaluación se ha fijado mediante `OOS_START = 2025-01-15` y `OOS_END = 2026-07-15`, el período efectivo del target termina antes, en la última fecha para la que existe un horizonte completo de 21 sesiones posteriores. Las observaciones situadas al final del período que todavía no disponen de dicho horizonte se excluyen automáticamente.

Finalmente, el target se alinea mediante el índice `(date, ticker)` con la matriz de características `X_oos`. De esta forma, cada observación utilizada posteriormente en la evaluación contiene simultáneamente las tres variables explicativas disponibles en la fecha de predicción y su correspondiente retorno realizado a 21 sesiones. Se eliminan las observaciones sin target disponible y se realizan comprobaciones de integridad para garantizar que `X_oos` e `y_oos` presentan exactamente las mismas observaciones y no contienen valores faltantes.

El resultado es el dataset `y_oos`, que queda congelado junto con `X_oos` y constituye la referencia de retornos realizados frente a la que se evaluarán las predicciones de los modelos finales.


In [14]:
from src.preprocessing.factor_preprocessing import compute_forward_return

X_oos = pd.read_parquet(
    "../data/model_results/final_model/oos/X_oos.parquet"
)


# =============================================================================
# Compute OOS Forward Returns
# =============================================================================

forward_return_21d = compute_forward_return(
    prices["Adj Close"],
    horizon=21,
)


# =============================================================================
# Build OOS Target Dataset
# =============================================================================

y_oos = (
    forward_return_21d
    .stack()
    .rename("forward_return_21d")
    .rename_axis(["date", "ticker"])
    .to_frame()
)


# =============================================================================
# Restrict to OOS Evaluation Period
# =============================================================================

oos_dates = y_oos.index.get_level_values("date")

y_oos = y_oos.loc[
    (oos_dates >= OOS_START)
    & (oos_dates <= OOS_END)
]


# =============================================================================
# Align Target with OOS Features
# =============================================================================

y_oos = y_oos.loc[
    y_oos.index.intersection(X_oos.index)
]


# =============================================================================
# Remove observations without realized forward return
# =============================================================================

y_oos = y_oos.dropna(
    subset=["forward_return_21d"]
)


# =============================================================================
# Final Alignment
# =============================================================================

common_index = X_oos.index.intersection(
    y_oos.index
)

X_oos = X_oos.loc[common_index]
y_oos = y_oos.loc[common_index]

In [15]:
X_oos.to_parquet(
    "../data/model_results/final_model/oos/X_oos.parquet"
)

print("X_oos successfully saved to: ../data/model_results/final_model/oos/X_oos.parquet")

X_oos successfully saved to: ../data/model_results/final_model/oos/X_oos.parquet


In [33]:
# =============================================================================
# Data Integrity Checks
# =============================================================================

print("\n" + "=" * 80)
print("OOS TARGET DATASET")
print("=" * 80)

print(
    "\nOOS period:",
    y_oos.index.get_level_values("date").min(),
    "→",
    y_oos.index.get_level_values("date").max(),
)

print(
    f"Observations: {len(y_oos):,}"
)

print(
    f"Unique tickers: "
    f"{y_oos.index.get_level_values('ticker').nunique():,}"
)

print("\nMissing values:")
print(
    y_oos.isna().sum()
)

print("\nTarget statistics:")
print(
    y_oos["forward_return_21d"].describe()
)



OOS TARGET DATASET

OOS period: 2025-01-15 00:00:00 → 2026-07-10 00:00:00
Observations: 184,408
Unique tickers: 496

Missing values:
forward_return_21d    0
dtype: int64

Target statistics:
count    184408.000000
mean          0.014141
std           0.106343
min          -0.558701
25%          -0.044250
50%           0.009537
75%           0.065378
max           2.086041
Name: forward_return_21d, dtype: float64


In [ ]:
print("\n" + "=" * 80)
print("X / y ALIGNMENT")
print("=" * 80)

print(
    f"X_oos observations: {len(X_oos):,}"
)

print(
    f"y_oos observations: {len(y_oos):,}"
)

assert X_oos.index.equals(y_oos.index), (
    "X_oos and y_oos do not have identical indices."
)

assert not y_oos.isna().any().any(), (
    "OOS target contains missing values."
)


print("\nX_oos and y_oos are perfectly aligned.")


X / y ALIGNMENT
X_oos observations: 184,408
y_oos observations: 184,408

X_oos and y_oos are perfectly aligned.


In [35]:
y_oos.to_parquet(
    "../data/model_results/final_model/oos/y_oos.parquet"
)

print("Y_oos successfully saved to: ../data/model_results/final_model/oos/y_oos.parquet")

Y_oos successfully saved to: ../data/model_results/final_model/oos/y_oos.parquet


## 4. Generate OOS Predictions

### 4.1 Generate predictions


In [16]:
X_oos = pd.read_parquet(
    "../data/model_results/final_model/oos/X_oos.parquet"
)

y_oos = pd.read_parquet(
    "../data/model_results/final_model/oos/y_oos.parquet"
)

# -----------------------------------------------------------------------------
# Models
# -----------------------------------------------------------------------------

models = {
    "ridge_rank": final_ridge_rank,
    "xgb_rank": final_xgb_rank,
    "rf_rank": final_rf_rank,
}


# =============================================================================
# Generate predictions
# =============================================================================

oos_predictions = X_oos.copy()

for name, model in models.items():

    oos_predictions[f"prediction_{name}"] = (
        model.predict(X_oos)
    )


# =============================================================================
# Add realized target
# =============================================================================

oos_predictions["forward_return_21d"] = (
    y_oos["forward_return_21d"]
)



In [17]:

# =============================================================================
# Inspect predictions
# =============================================================================

print("\n" + "=" * 80)
print("OOS PREDICTIONS")
print("=" * 80)

print(
    f"\nObservations: {len(oos_predictions):,}"
)

print(
    f"Unique tickers: "
    f"{oos_predictions.index.get_level_values('ticker').nunique():,}"
)

print("\nPrediction columns:")

print(
    [
        col
        for col in oos_predictions.columns
        if col.startswith("prediction_")
    ]
)

print("\nMissing values:")

print(
    oos_predictions[
        [
            "prediction_ridge_rank",
            "prediction_xgb_rank",
            "prediction_rf_rank",
            "forward_return_21d",
        ]
    ].isna().sum()
)

print("\nPrediction statistics:")

print(
    oos_predictions[
        [
            "prediction_ridge_rank",
            "prediction_xgb_rank",
            "prediction_rf_rank",
        ]
    ].describe()
)

print("\n" + "=" * 80)


OOS PREDICTIONS

Observations: 184,408
Unique tickers: 496

Prediction columns:
['prediction_ridge_rank', 'prediction_xgb_rank', 'prediction_rf_rank']

Missing values:
prediction_ridge_rank    0
prediction_xgb_rank      0
prediction_rf_rank       0
forward_return_21d       0
dtype: int64

Prediction statistics:
       prediction_ridge_rank  prediction_xgb_rank  prediction_rf_rank
count          184408.000000        184408.000000       184408.000000
mean                0.014807             0.014821            0.015472
std                 0.004190             0.003660            0.009112
min                 0.005012             0.013136            0.011264
25%                 0.011615             0.013136            0.012054
50%                 0.014853             0.013318            0.013353
75%                 0.017929             0.015174            0.015931
max                 0.024854             0.057154            0.159783



In [18]:
# =============================================================================
# Prediction Integrity Checks
# =============================================================================

prediction_cols = [
    "prediction_ridge_rank",
    "prediction_xgb_rank",
    "prediction_rf_rank",
]

assert oos_predictions.index.equals(
    X_oos.index
)

assert not oos_predictions[
    prediction_cols
].isna().any().any()

assert not oos_predictions[
    "forward_return_21d"
].isna().any()

print("All OOS predictions generated successfully.")
print("All models evaluated on the same OOS observations.")

All OOS predictions generated successfully.
All models evaluated on the same OOS observations.


In [19]:
missing_target = oos_predictions[
    oos_predictions["forward_return_21d"].isna()
]

print(missing_target.index.get_level_values("date").min())
print(missing_target.index.get_level_values("date").max())

print(
    missing_target
    .groupby(level="ticker")
    .size()
    .sort_values(ascending=False)
)

NaT
NaT
Series([], dtype: int64)


   ### 4.2 Store predictions
   ### 4.3 Cross-sectional prediction analysis

## 5. OOS Predictive Performance
   ### 5.1 RMSE / MAE
   ### 5.2 Information Coefficient (IC) & Rank IC
   ### 5.3 IC Stability (IC Sharpe Ratio)
   ### 5.4 Hit Rate / Directional Accuracy
   ### 5.5 Temporal evolution of predictive power

## 6. OOS Diagnostics
   ### 6.1 Prediction distribution
   ### 6.2 Predicted vs. realized returns
   ### 6.3 Cross-sectional quantile/decile analysis (Monotonicity check)
   ### 6.4 Factor/model stability

## 7. Export OOS Results
   ### 7.1 OOS predictions
   ### 7.2 OOS metrics
   ### 7.3 OOS metadata

## 8. Conclusions